In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
RANDOM_STATE = 123 # constant for reproducibility

In [2]:
# repeat steps in EDA to format dataframe correctly
netflix_df = pd.read_csv('netflix_user_behavior_dataset.csv')

netflix_df['user_id'] = netflix_df['user_id'].astype(str)
netflix_df['age'] = netflix_df['age'].astype(int)
netflix_df['gender'] = netflix_df['gender'].astype('category')
netflix_df['country'] = netflix_df['country'].astype('category')
netflix_df['account_age_months'] = netflix_df['account_age_months'].astype(int)
netflix_df['subscription_type'] = netflix_df['subscription_type'].astype('category')
netflix_df['monthly_fee'] = netflix_df['monthly_fee'].astype(float)
netflix_df['payment_method'] = netflix_df['payment_method'].astype('category')
netflix_df['primary_device'] = netflix_df['primary_device'].astype('category')
netflix_df['devices_used'] = netflix_df['devices_used'].astype(int)
netflix_df['favorite_genre'] = netflix_df['favorite_genre'].astype('category')
netflix_df['avg_watch_time_minutes'] = netflix_df['avg_watch_time_minutes'].astype(int)
netflix_df['watch_sessions_per_week'] = netflix_df['watch_sessions_per_week'].astype(int)
netflix_df['binge_watch_sessions'] = netflix_df['binge_watch_sessions'].astype(int)
netflix_df['completion_rate'] = netflix_df['completion_rate'].astype(int)
netflix_df['rating_given'] = netflix_df['rating_given'].astype(float)
netflix_df['content_interactions'] = netflix_df['content_interactions'].astype(int)
netflix_df['recommendation_click_rate'] = netflix_df['recommendation_click_rate'].astype(int)
netflix_df['days_since_last_login'] = netflix_df['days_since_last_login'].astype(int)

churn_map = {'No': False, 'Yes': True}
netflix_df['churned'] = netflix_df['churned'].map(churn_map).astype(bool)

In [3]:
# separate into train and test
predictors = netflix_df.drop(columns=['user_id', 'churned'])
target = netflix_df['churned']
X_train, X_test, y_train, y_test = train_test_split(predictors, target, test_size=0.2, random_state=RANDOM_STATE) # use random state constant for reproducibility

In [4]:
# Convert categorical predictors into numbers
X = pd.get_dummies(
    netflix_df.drop(columns=['user_id', 'churned']),
    drop_first=True
)

y = netflix_df['churned']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

Categorical variables transformed to ensure compatibility with machine learning algorithms that require numeric input.

In [7]:
# Baseline Model - Logistic Regression
from sklearn.linear_model import LogisticRegression

log_model_bal = LogisticRegression(max_iter=5000)

# train
log_model_bal.fit(X_train, y_train)

# predict
log_bal_preds = log_model_bal.predict(X_test)

In [8]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, log_bal_preds))
print(classification_report(y_test, log_bal_preds))

[[8010    0]
 [1990    0]]
              precision    recall  f1-score   support

       False       0.80      1.00      0.89      8010
        True       0.00      0.00      0.00      1990

    accuracy                           0.80     10000
   macro avg       0.40      0.50      0.44     10000
weighted avg       0.64      0.80      0.71     10000



c:\Users\hanna\miniconda3\envs\ads502\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hanna\miniconda3\envs\ads502\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hanna\miniconda3\envs\ads502\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

In [5]:
# Baseline Model - Logistic Regression
from sklearn.linear_model import LogisticRegression

log_model_bal = LogisticRegression(max_iter=5000, class_weight='balanced')

# train
log_model_bal.fit(X_train, y_train)

# predict
log_bal_preds = log_model_bal.predict(X_test)

In [22]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, log_bal_preds))
print(classification_report(y_test, log_bal_preds))

[[4078 3932]
 [1005  985]]
              precision    recall  f1-score   support

       False       0.80      0.51      0.62      8010
        True       0.20      0.49      0.29      1990

    accuracy                           0.51     10000
   macro avg       0.50      0.50      0.45     10000
weighted avg       0.68      0.51      0.56     10000



When using logistic regression, the model initially predicted all users as non-churn, resulting in an accuracy of 80% but failing to identify any churned users. After applying class weighting, the model improved its ability to detect churn, with recall increasing from 0.00 to approximately 0.49. However, this improvement came at the cost of lower precision and overall accuracy, indicating that the model produced more false positives while better identifying churned users.

In [25]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree_model.fit(X_train, y_train)

tree_preds = tree_model.predict(X_test)

In [26]:
print(confusion_matrix(y_test, tree_preds))
print(classification_report(y_test, tree_preds))

[[6265 1745]
 [1531  459]]
              precision    recall  f1-score   support

       False       0.80      0.78      0.79      8010
        True       0.21      0.23      0.22      1990

    accuracy                           0.67     10000
   macro avg       0.51      0.51      0.51     10000
weighted avg       0.69      0.67      0.68     10000



The Decision Tree model achieved higher overall accuracy (67%) compared to logistic regression. However, when identifying churned users, its recall was lower than the logistic regression model, correctly identifying only 23% of churned users. The class-weighted logistic regression model detected churn with a higher recall measure of near 49%, but sacrifices accuracy. This conveys the tradeoffs between the ability to correctly identify churned customers and maximizing model accuracy.

In [ ]:
# test
